In [71]:
import datetime
import numpy as np

class User:
    def __init__(self, user_id: int, name: str, email: str):
        self._id = user_id
        self._name = name.strip().title()
        if '@' not in email:
            raise ValueError('Not an email')
        self._email = email.strip().lower()

    @classmethod
    def from_string(cls, data: str):
        parts = [part.strip() for part in data.split(',')]
        if len(parts) != 3:
            raise ValueError('Line is less than 3 or more')
        return cls(int(parts[0]), parts[1], parts[2])

    def __str__(self):
        return f"User(id={self._id}, name='{self._name}', email='{self._email}')"

    def __del__(self):
        print(f"user {self._name} deleted")

class Product:
    def __init__(self, product_id: int, name: str, price: float, category: str):
        self.id = product_id
        self.name = name
        self.price = price
        self.category = category

    def __repr__(self):
        return f"Product('{self.name}', {self.price})"

    def __str__(self):
        return f"Product(id={self.id}, name='{self.name}', price={self.price}, category='{self.category}')"

    def __eq__(self, other):
        if not isinstance(other, Product):
            return False
        return self.id == other.id

    def __hash__(self):
        return hash(self.id)

    def to_dict(self):
        return {
            'id': self.id,
            'name': self.name,
            'price': self.price,
            'category': self.category
        }

class Inventory:
    def __init__(self):
        self._products = {}

    def add_product(self, product: Product):
        self._products[product.id] = product

    def remove_product(self, product_id: int):
        self._products.pop(product_id, None)

    def get_product(self, product_id: int):
        return self._products.get(product_id)

    def get_all_products(self):
        return list(self._products.values())

    def unique_products(self):
        return set(self._products.values())

    def to_dict(self):
        return self._products

    def filter_by_price(self, min_price: float) -> list[Product]:
        return [p for p in self.get_all_products() if (lambda x: x >= min_price)(p.price)]

class Logger:
    @staticmethod
    def log_action(user: User, action: str, product: Product, filename: str):
        timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        log_entry = f"{timestamp};{user._id};{action};{product.id}\n"
        with open(filename, 'a', encoding='utf-8') as f:
            f.write(log_entry)

    @staticmethod
    def read_logs(filename: str):
        logs = []
        try:
            with open(filename, 'r', encoding='utf-8') as f:
                for line in f:
                    parts = line.strip().split(';')
                    logs.append({
                        'timestamp': parts[0],
                        'user_id': parts[1],
                        'action': parts[2],
                        'product_id': parts[3]
                    })
        except FileNotFoundError:
            return []
        return logs

class Order:
    def __init__(self, order_id: int, user: User, products=None):
        self.id = order_id
        self.user = user
        self.products = products if products else []

    def add_product(self, product: Product):
        self.products.append(product)

    def remove_product(self, product_id: int):
        self.products = [p for p in self.products if p.id != product_id]

    def total_price(self):
        return sum(p.price for p in self.products)

    def __str__(self):
        return f'Order #{self.id} for {self.user._name}, Total: {self.total_price()}'

    def most_expensive_products(self, n: int) -> list[Product]:
        return sorted(self.products, key=lambda p: p.price, reverse=True)[:n]

def create_orders_matrix(orders_list):
    return np.array([[o.total_price()] for o in orders_list])

def price_stream(products: list[Product]):
    for p in products:
        yield p.price

class OrderIterator:
    def __init__(self, orders: list[Order]):
        self._orders = orders
        self._index = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self._index < len(self._orders):
            res = self._orders[self._index]
            self._index += 1
            return res
        raise StopIteration

In [72]:
#11
def get_prices(products):
    return np.array([p.price for p in products],dtype=float)
produsto=[Product(1,"Laptop",1200.0,"Electronics"), Product(2,"Mouse",25.0,"Electronics")]
prices=get_prices(produsto)
print(prices)

[1200.   25.]


In [73]:
#12
def stats(price):
    return (round(np.mean(price),2),np.median(price))
print(stats(np.array([1200,25,450])))

(np.float64(558.33), np.float64(450.0))


In [74]:
#13
def normalize(prices):
    p_min=prices.min()
    p_max=prices.max()
    return np.round((prices-p_min)/(p_max-p_min),4)
ppp=np.array([1200,25,450])
print(normalize(ppp))

[1.     0.     0.3617]


In [75]:
#14
def get_category(products):
    return [p.category for p in products]
products = [Product(1,"Laptop",1200.0,"Electronics"), Product(2,"T-Shirt",20.0,"Clothing")]
print(get_category(products))

['Electronics', 'Clothing']


In [76]:
#15
def counting(category):
    return len(np.unique(category))

cat=["Electronics", "Clothing", "Electronics"]
print(counting(cat))

2


In [77]:
#16
def expensive(products):
    prices=np.array([p.price for p in products])
    mmm=np.mean(prices)
    return [p for p in products if p.price>mmm]
p_list = [Product(1,"A",1200.0,"E"), Product(2,"B",25.0,"E"), Product(3,"C",450.0,"E")]
print([p.name for p in expensive(p_list)])

['A']


In [78]:
#17
def degrree(price):
    return price*0.9
pr=np.array([1200.0, 25.0, 450.0])
print(degrree(pr))

[1080.    22.5  405. ]


In [79]:
#18
def create_orders(orderrr):
    sums_2d = [[order.total_price()] for order in orderrr]
    return np.array(sums_2d)

u1 = User(1, "Alice",'@ertyujkjhg')
u2 = User(2, "Bob",'@dfghjkb')


p1 = Product(1, "Laptop", 1200.0, "Electronics")
p2 = Product(2, "Mouse", 25.0, "Electronics")


orders = [
    Order(1, u1, [p1]),
    Order(2, u2, [p2, p1])
]
matrix = create_orders(orders)
print(matrix)

user Bob deleted
user Alice deleted
[[1200.]
 [1225.]]


In [80]:
#19
def average(prices):
    return np.mean(prices)
ppp=np.array([1200.0, 1225.0])
print(average(ppp))

1212.5


In [81]:
#20
def highest(matrix):
    return np.where(matrix>1000)[0].tolist()

prices = np.array([1200.0, 900.0, 1500.0])
print(highest(prices))

[0, 2]


In [82]:
#21
import pandas as pd
from datetime import date
def create_list(user_list):
    data=[{
        'id':u._id,
        'name':u._email,
        'registration_date':date.today().isoformat()
    } for u in user_list]
    return pd.DataFrame(data)
users = [User(1, "John Doe", "john@example.com"), User(2, "Alice", "alice@example.com")]
print(create_list(users))

   id               name registration_date
0   1   john@example.com        2026-04-25
1   2  alice@example.com        2026-04-25


In [83]:
#22
def create_list_products(products):
    data=[{
        'product-id':p.id,
        'product-name':p.name,
        'price':p.price,
        'category':p.category
    } for p in products]
    return pd.DataFrame(data)
products = [Product(1,"Laptop",1200.0,"Electronics"), Product(2,"T-Shirt",20.0,"Clothing")]
print(create_list_products(products))

   product-id product-name   price     category
0           1       Laptop  1200.0  Electronics
1           2      T-Shirt    20.0     Clothing


In [84]:
#23
def merge_orders(u_df,o_df):
    merged=pd.merge(o_df,u_df[['id','name']],left_on='user_id',right_on='id')
    result=merged[['order_id','name','total']].rename(columns={'name':'user_name'})
    return result
u_df = pd.DataFrame({'id': [1, 2], 'name': ['John', 'Alice']})
o_df = pd.DataFrame({'order_id': [101, 102], 'user_id': [1, 2], 'total': [1200, 25]})
print(merge_orders(u_df,o_df))

   order_id user_name  total
0       101      John   1200
1       102     Alice     25


In [86]:
#24
def filterr(df,min_total):
    return df[df['total']>min_total]

print(filterr(merge_orders(u_df,o_df),100))

   order_id user_name  total
0       101      John   1200


In [87]:
#25
def groupping(df):
    return df.groupby('user_name')['total'].sum().reset_index(name='total_sum')

data = {'user_name': ['John', 'John', 'Alice'], 'total': [1200, 500, 25]}
df_orders = pd.DataFrame(data)
print(groupping(df_orders))

  user_name  total_sum
0     Alice         25
1      John       1700


In [88]:
#26
def groupping_price(df):
    return df.groupby('user_name')['total'].mean().reset_index(name='average_price')
print(groupping_price(df_orders))

  user_name  average_price
0     Alice           25.0
1      John          850.0


In [89]:
#27
def groupping_count(df):
    return df.groupby('user_name')['total'].count().reset_index(name='orders_count')
print(groupping_count(df_orders))

  user_name  orders_count
0     Alice             1
1      John             2


In [90]:
#28
def groupping_category(df):
    return df.groupby('category')['price'].mean().reset_index(name='mean_price')
data=pd.DataFrame({
    'id': [1, 2, 3],
    'name': ['Laptop', 'Mouse', 'Shirt'],
    'category': ['Electronics', 'Electronics', 'Clothing'],
    'price': [1200, 25, 20]
})
print(groupping_category(data))

      category  mean_price
0     Clothing        20.0
1  Electronics       612.5


In [91]:
#29
def add_column(df):
    dff=df.copy()
    dff['discount_price']=dff['price']*0.9
    return dff
p_simple = pd.DataFrame({'id': [1, 2], 'name': ['Laptop', 'Mouse'], 'price': [1200, 25]})
print(add_column(p_simple))

   id    name  price  discount_price
0   1  Laptop   1200          1080.0
1   2   Mouse     25            22.5


In [92]:
#30
def sorting(df):
    dff=df.copy()
    return dff.sort_values(by='price',ascending=False)
print(sorting(data))

   id    name     category  price
0   1  Laptop  Electronics   1200
1   2   Mouse  Electronics     25
2   3   Shirt     Clothing     20


In [93]:
#31
def add_column_quantity(df):
    df['quatity']=1
    return df
p_simple = pd.DataFrame({'id': [1, 2], 'name': ['Laptop', 'Mouse'], 'price': [1200, 25]})
print(add_column_quantity(p_simple))

   id    name  price  quatity
0   1  Laptop   1200        1
1   2   Mouse     25        1


In [94]:
#32
def calculate(df):
    df['total_price']=df['price']*df['quantity']
    return df
df32 = pd.DataFrame({'order_id': [101, 102], 'price': [1200, 25], 'quantity': [1, 2]})
print(calculate(df32))

   order_id  price  quantity  total_price
0       101   1200         1         1200
1       102     25         2           50


In [96]:
#33
def filterrr(df,category):
    return df[df['category']==category]
df33 = pd.DataFrame({'product_name': ['Laptop', 'T-Shirt'], 'category': ['Electronics', 'Clothing'], 'price': [1200, 20]})
print(filterrr(df33,'Clothing'))

  product_name  category  price
1      T-Shirt  Clothing     20


In [97]:
#34
def counting(df):
    return df.groupby('category').size().reset_index(name='count')

df34 = pd.DataFrame({'product_name': ['Laptop', 'Mouse', 'Shirt'], 'category': ['Electronics', 'Electronics', 'Clothing']})
print(counting(df34))

      category  count
0     Clothing      1
1  Electronics      2


In [99]:
#35
def average_price(df):
    return df.groupby('category')['price'].mean().reset_index(name='mean_price')

df35 = pd.DataFrame({'category': ['Electronics', 'Electronics', 'Clothing'], 'price': [1200, 25, 20]})
print(average_price(df35))

      category  mean_price
0     Clothing        20.0
1  Electronics       612.5


In [100]:
#36
def sorting(df):
    return df.sort_values(by='total_price',ascending=False)
df36 = pd.DataFrame({'order_id': [101, 102], 'total_price': [1200, 50]})
print(sorting(df36))

   order_id  total_price
0       101         1200
1       102           50


In [101]:
#37
def top_n(df,n):
    return df.sort_values(by='total_price',ascending=False).head(n)

df37 = pd.DataFrame({'order_id': [101, 102, 103, 104], 'total_price': [1200, 50, 500, 1500]})
print(top_n(df37, 3))

   order_id  total_price
3       104         1500
0       101         1200
2       103          500


In [103]:
#38
def merges(utff,otff):
    merged=pd.merge(utff,otff,on='user_id')
    return merged
utff=pd.DataFrame({'user_id':[1,2],'user_name':['Elya','Alena']})
otff=pd.DataFrame({'order_id':[101,102],'user_id':[1,2],'total_price':[1200,50]})
print(merges(utff,otff))

   user_id user_name  order_id  total_price
0        1      Elya       101         1200
1        2     Alena       102           50


In [ ]:
#39
